In [2]:
%cd ../

/home/student/pubtrends


In [3]:
!pwd

/home/student/pubtrends


In [4]:
import logging

import numpy as np
import pandas as pd

from pysrc.papers.db.loader import Loader
from pysrc.papers.db.postgres_connector import PostgresConnector
from pysrc.papers.db.postgres_utils import preprocess_search_query_for_postgres, \
    process_bibliographic_coupling_postgres, process_cocitations_postgres, no_stemming_filter
from pysrc.papers.utils import crc32, SORT_MOST_CITED, SORT_MOST_RECENT, preprocess_doi, \
    preprocess_search_title

logger = logging.getLogger(__name__)

import logging
import html
import pandas as pd
import numpy as np
import networkx as nx
import random
import hashlib
from tqdm.auto import tqdm

from pysrc.prediction.ss_arxiv_loader import SSArxivLoader
from pysrc.prediction.ss_pubmed_loader import SSPubmedLoader
from pysrc.papers.db.pm_postgres_loader import PubmedPostgresLoader
from pysrc.prediction.predict_analyzer import PredictAnalyzer
from pysrc.papers.config import PubtrendsConfig
from pysrc.papers.db.ss_postgres_loader import SemanticScholarPostgresLoader
from pysrc.papers.db.postgres_connector import PostgresConnector
from collections import defaultdict



class CustomLoader(SemanticScholarPostgresLoader):
    def __init__(self, config):
        super(CustomLoader, self).__init__(config)

    def load_func(self, limit=100):
        self.check_connection()
        if limit is None:
            query = '''SELECT ssid, aux::json->'authors' FROM sspublications'''
        else:
            query = f'''
                    SELECT ssid, aux::json->'authors' FROM sspublications LIMIT {limit}
            '''
        result = defaultdict(list)
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            for item in cursor:
                ind, names = item
                for i, el in enumerate(names):
                    result[el['name']].append((ind, int(i == 0)))
            return result
    
    def custom_query(self, query):
        self.check_connection()
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            return cursor.fetchall()
    
    def create_subsample(self, threshold=0.01, seed=42):
        self.check_connection()
        random.seed(seed)
        ssids, crc32ids = [], []
        query = '''select ssid, crc32id from sspublications'''
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            for item in cursor:
                if random.random() < threshold:
                    ss, crc = item
                    ssids.append(ss)
                    crc32ids.append(crc)
            return ssids, crc32ids


class CustomWriter(PostgresConnector):

    def __init__(self, config):
        super(CustomWriter, self).__init__(config, readonly=False)
        
    def insert_table_publications(self, ids):
        self.check_connection()
        query = f'''CREATE TABLE IF NOT EXISTS sspublications_sample AS 
            (SELECT * FROM sspublications WHERE ssid IN ({', '.join(map(lambda x: "'" + str(x) + "'", ids))}))'''
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            self.postgres_connection.commit()
        
    def insert_table_citations(self, ids):
        self.check_connection()
        query = f'''CREATE TABLE IF NOT EXISTS sscit_sample AS 
            (SELECT * FROM sscitations WHERE crc32id_in IN ({', '.join(map(lambda x: "'" + str(x) + "'", ids))}));
            create index if not exists sscit_crc32id_in on sscit_sample (crc32id_in);'''
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            self.postgres_connection.commit()
    
    def execute_custom_query(self, query):
        try:
            self.check_connection()
            with self.postgres_connection.cursor() as cursor:
                cursor.execute(query)
                self.postgres_connection.commit()
        except Exception as e:
            print(e)


In [5]:
config = PubtrendsConfig(test=False)
loader = CustomLoader(config)
writer = CustomWriter(config)

### Selecting 1% subsample for experiments

In [17]:
%%time
ssids, crcids = loader.create_subsample(threshold=0.01)

CPU times: user 47.6 s, sys: 7.68 s, total: 55.3 s
Wall time: 4min 38s


In [6]:
%%time
writer.insert_table_publications(ssids)

CPU times: user 390 ms, sys: 63.1 ms, total: 453 ms
Wall time: 9min 32s


In [18]:
%%time
writer.insert_table_citations(crcids)

CPU times: user 47.1 ms, sys: 9.37 ms, total: 56.4 ms
Wall time: 28min 47s


### Features calculating

### Create table for authors_papers

In [99]:
import string
import zlib


punc = set(string.punctuation)


def process_name(st):
    return ''.join([item for item in st if item not in punc]).strip()


def select_authors():
    data = defaultdict(list)
    loader.check_connection()
    query = '''SELECT ssid, crc32id, year, aux::json->'authors' FROM sspublications_sample'''
    with loader.postgres_connection.cursor() as cursor:
        cursor.execute(query)
        for item in cursor:
            ind, crc, year, names = item
            for i, el in enumerate(names):
                data[process_name(el['name'])].append((ind, crc, year, int(i == 0)))
        del data['']
        return data
    

def process_data(data):
    results = []
    tmp = {True: 'TRUE', False: 'FALSE'}
    for idx, item in enumerate(data, start=1):
        if item:
            for ind, crc, year, is_main in data[item]:
                results.append(f'''({idx}, '{ind}', {crc}, {year if year is not None else "NULL"}, '{item[:100]}', {tmp[is_main]})''')
    return results


def insert_authors_papers_data(data):
    try:
        writer.check_connection()
        query_init = '''CREATE TABLE IF NOT EXISTS authors_papers (
                            author_id bigint not null, 
                            ssid varchar(40) not null,
                            crc32id_paper integer not null,
                            year integer,
                            author_name varchar(100) not null, 
                            is_main boolean);
                        create index if not exists authors_papers_id_author_year on authors_papers (author_id, year);'''
        
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query_init)
            writer.postgres_connection.commit()
            
        with writer.postgres_connection.cursor() as cursor:
            query_insert = f'''insert into authors_papers(author_id, ssid, crc32id_paper, year, author_name, is_main) values {','.join(data)}'''
            cursor.execute(query_insert)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)


    

In [100]:
%%time
df = select_authors()
data = process_data(df)
insert_authors_papers_data(data)

CPU times: user 21 s, sys: 1.4 s, total: 22.4 s
Wall time: 1min 22s


### Create table for authors

In [102]:
def create_authors_table():
    query = '''create table if not exists authors as (select distinct on (author_id) author_id, author_name from authors_papers);
                create index if not exists authots_id_index on authors (author_id);'''
    writer.execute_custom_query(query=query)


In [103]:
%%time
create_authors_table()

CPU times: user 1.87 ms, sys: 236 µs, total: 2.1 ms
Wall time: 3.93 s


### Calculate productivity


In [33]:
%%time
query_prod = '''alter table authors add column if not exists productivity decimal default 0.0;
                    update authors set productivity = t.prod 
                        from (select author_id, coalesce(cast(count(*) as decimal) / (2021 - min(year) + 1), 0.0) as prod 
                            from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_prod)

CPU times: user 1.7 ms, sys: 148 µs, total: 1.85 ms
Wall time: 14.4 s


### Calculate max and min year of publications

In [106]:
%%time
query_years = '''alter table authors add column if not exists first_year_pub int;
                    alter table authors add column if not exists last_year_pub int;
                    update authors set first_year_pub = t.res 
                        from (select author_id, min(year) as res 
                            from authors_papers group by author_id) as t where t.author_id = authors.author_id;
                    update authors set last_year_pub = t.res 
                        from (select author_id, max(year) as res 
                            from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_years)

CPU times: user 2.45 ms, sys: 0 ns, total: 2.45 ms
Wall time: 27.4 s


### Calculate total papers

In [30]:
%%time
query_papers = '''alter table authors add column if not exists total_papers int default 0;
                    update authors set total_papers = t.res 
                        from (select author_id, count(*) as res 
                            from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_papers)

CPU times: user 1.5 ms, sys: 130 µs, total: 1.63 ms
Wall time: 14.3 s


### Calculate ratio of main authering


In [31]:
%%time
query_is_main = '''alter table authors add column if not exists is_main_ratio decimal default 0.0;
                    update authors set is_main_ratio = t.res 
                        from (select author_id, cast(count(case when is_main then 1 end) as decimal) / count(*) as res
                        from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_is_main)

CPU times: user 1.54 ms, sys: 133 µs, total: 1.67 ms
Wall time: 15.4 s


### Calculate avg num of coauthors

In [32]:
%%time
query_coauthors = '''alter table authors add column if not exists sociality decimal default 0.0;
                    update authors set sociality = t.res 
                        from (select author_id, avg(count - 1) as res
                            from (select * from authors_papers left join (select crc32id_paper, count(*)
                                from authors_papers group by crc32id_paper) as f on authors_papers.crc32id_paper=f.crc32id_paper) as tab group by tab.author_id) 
                                as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_coauthors)

CPU times: user 1.78 ms, sys: 154 µs, total: 1.93 ms
Wall time: 17.6 s


### Calculcate total num of venues

In [110]:
%%time
q = '''select authors_papers.author_id, ven, jour from authors_papers left join (SELECT ssid, aux::json->'venue' as ven, aux::json#>'{journal, name}' as jour
                                        FROM sspublications_sample) as t on authors_papers.ssid = t.ssid'''
journals_and_venues = loader.custom_query(q)

CPU times: user 13.5 s, sys: 309 ms, total: 13.8 s
Wall time: 4min 4s


In [111]:
def calc_total_venues(data):
    try:
        authors = {}
        for a_id, ven, jour in data:
            if a_id not in authors:
                authors[a_id] = set()
            if ven:
                authors[a_id].add(ven)
            if jour:
                authors[a_id].add(jour)
        ls = []
        for key in authors:
            ls.append(f'''({key}, {len(authors[key])})''')
        
        query = f'''
                    alter table authors add column if not exists total_venues int default 0;
                    create table if not exists temp_table (a_id bigint, venues int);
                    insert into temp_table values {','.join(ls)};
                    update authors set total_venues = t.venues 
                        from (select a_id, venues from temp_table) as t where t.a_id = authors.author_id;
                    drop table temp_table;
                    '''
        writer.check_connection()
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)
        
        

In [112]:
%%time
calc_total_venues(journals_and_venues)

CPU times: user 3.85 s, sys: 322 ms, total: 4.17 s
Wall time: 24.8 s


### Calculate citations (c2, c5, c_all)

In [35]:
# all
# select crc32id_in, count(crc32id_out) from sscit_sample group by crc32id_in;

# alter table sscit_sample add column if not exists out_year int;
# alter table sscit_sample add column if not exists in_year int;
# update sscit_sample set out_year = t.year from (select crc32id, year from sspublications) as t where t.crc32id = sscit_sample.crc32id_out;
# update sscit_sample set in_year = t.year from (select crc32id, year from sspublications_sample) as t where t.crc32id = sscit_sample.crc32id_in;

# c2
# select crc32id_in, sum(case when out_year - in_year <= 2 and  out_year - in_year >= 0 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;
# alter table sspublications_sample add column if not exists c2 int;
# update sspublications_sample set c2 = t.res from (select crc32id_in, sum(case when out_year - in_year <= 2 and  out_year - in_year >= 0 then 1 else 0 end) as res from sscit_sample group by sscit_sample.crc32id_in) as t where t.crc32id_in = sspublications_sample.crc32id;

# c5
# select crc32id_in, sum(case when out_year - in_year <= 5 and  out_year - in_year >= 0 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_all
# select crc32id_in, sum(case when out_year - in_year >= 0 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;


In [113]:
%%time
query_in_year_and_out_year = '''
                alter table sscit_sample add column if not exists out_year int;
                alter table sscit_sample add column if not exists in_year int;
                update sscit_sample set out_year = t.year from (select ssid, year from sample_ext) as t where t.ssid = sscit_sample.ssid_out;
                update sscit_sample set in_year = t.year from (select ssid, year from sspublications_sample) as t where t.ssid = sscit_sample.ssid_in;
                '''
writer.execute_custom_query(query=query_in_year_and_out_year)

CPU times: user 16 ms, sys: 954 µs, total: 16.9 ms
Wall time: 5min 18s


In [5]:
%%time
query_c2 = '''
                alter table sspublications_sample add column if not exists c2 int default 0;
                update sspublications_sample set c2 = t.res 
                    from (select ssid_in, sum(case when out_year - in_year <= 2 and  out_year - in_year >= 0 then 1 else 0 end) as res 
                        from sscit_sample group by sscit_sample.ssid_in) as t
                            where t.ssid_in = sspublications_sample.ssid;
                '''
writer.execute_custom_query(query=query_c2)

CPU times: user 0 ns, sys: 3.95 ms, total: 3.95 ms
Wall time: 19.1 s


In [6]:
%%time
query_c5 = '''
                alter table sspublications_sample add column if not exists c5 int default 0;
                update sspublications_sample set c5 = t.res 
                    from (select ssid_in, sum(case when out_year - in_year <= 5 and  out_year - in_year >= 0 then 1 else 0 end) as res 
                        from sscit_sample group by sscit_sample.ssid_in) as t
                            where t.ssid_in = sspublications_sample.ssid;
            '''
writer.execute_custom_query(query=query_c5)

CPU times: user 1.69 ms, sys: 197 µs, total: 1.88 ms
Wall time: 16.7 s


In [7]:
%%time
query_c_all = '''
                alter table sspublications_sample add column if not exists c_all int default 0;
                update sspublications_sample set c_all = t.res 
                    from (select ssid_in, sum(case when out_year - in_year >= 0 then 1 else 0 end) as res 
                        from sscit_sample group by sscit_sample.ssid_in) as t
                            where t.ssid_in = sspublications_sample.ssid;
                '''
writer.execute_custom_query(query=query_c_all)

CPU times: user 2.08 ms, sys: 243 µs, total: 2.33 ms
Wall time: 16.3 s


### Caclculate mean citations for authors

In [8]:
%%time
query_mean_citations = '''
                        alter table authors add column if not exists mean_citations decimal;
                        update authors set mean_citations = t.res 
                                            from (select f.author_id, avg(f.c_all) as res from (select author_id, c_all 
                                            from authors_papers left join sspublications_sample on authors_papers.ssid = sspublications_sample.ssid) as f 
                                            group by f.author_id) as t where t.author_id = authors.author_id;
                        '''
writer.execute_custom_query(query=query_mean_citations)

CPU times: user 0 ns, sys: 3.43 ms, total: 3.43 ms
Wall time: 17.7 s


### Calculate top cited paper for authors

In [9]:
%%time
query_top_citatoons = '''
                        alter table authors add column if not exists top_citations int;
                        update authors set top_citations = t.res 
                            from (select f.author_id, max(f.c_all) as res from (select author_id, c_all 
                            from authors_papers left join sspublications_sample on authors_papers.ssid = sspublications_sample.ssid) as f
                            group by f.author_id) as t where t.author_id = authors.author_id;
                        '''
writer.execute_custom_query(query=query_top_citatoons)

CPU times: user 1.76 ms, sys: 216 µs, total: 1.98 ms
Wall time: 16.7 s


### Calculate h-index

In [ ]:
# c_1970
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1970 and in_year <= 1970 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_1980
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1980 and in_year <= 1980 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_1990
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1990 and in_year <= 1990 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_2000
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2000 and in_year <= 2000 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_2010
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2010 and in_year <= 2010 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_2020
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2020 and in_year <= 2020 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

In [15]:
%%time 
query_c_1970 = '''
                alter table sspublications_sample add column if not exists c_1970 int default 0;
                update sspublications_sample set c_1970 = t.res
                    from (select ssid_in, sum(case when out_year - in_year >= 0 and out_year <= 1970 and in_year <= 1970 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.ssid_in) as t where t.ssid_in = sspublications_sample.ssid;
'''
writer.execute_custom_query(query=query_c_1970)

CPU times: user 2.21 ms, sys: 279 µs, total: 2.48 ms
Wall time: 17.7 s


In [16]:
%%time 
query_c_1980 = '''
                alter table sspublications_sample add column if not exists c_1980 int default 0;
                update sspublications_sample set c_1980 = t.res
                    from (select ssid_in, sum(case when out_year - in_year >= 0 and out_year <= 1980 and in_year <= 1980 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.ssid_in) as t where t.ssid_in = sspublications_sample.ssid;
'''
writer.execute_custom_query(query=query_c_1980)

CPU times: user 1.56 ms, sys: 196 µs, total: 1.75 ms
Wall time: 16.7 s


In [17]:
%%time 
query_c_1990 = '''
                alter table sspublications_sample add column if not exists c_1990 int default 0;
                update sspublications_sample set c_1990 = t.res
                    from (select ssid_in, sum(case when out_year - in_year >= 0 and out_year <= 1990 and in_year <= 1990 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.ssid_in) as t where t.ssid_in = sspublications_sample.ssid;
'''
writer.execute_custom_query(query=query_c_1990)

CPU times: user 1.65 ms, sys: 0 ns, total: 1.65 ms
Wall time: 18.6 s


In [18]:
%%time 
query_c_2000 = '''
                alter table sspublications_sample add column if not exists c_2000 int default 0;
                update sspublications_sample set c_2000 = t.res
                    from (select ssid_in, sum(case when out_year - in_year >= 0 and out_year <= 2000 and in_year <= 2000 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.ssid_in) as t where t.ssid_in = sspublications_sample.ssid;
'''
writer.execute_custom_query(query=query_c_2000)

CPU times: user 2.01 ms, sys: 0 ns, total: 2.01 ms
Wall time: 21.7 s


In [19]:
%%time 
query_c_2010 = '''
                alter table sspublications_sample add column if not exists c_2010 int default 0;
                update sspublications_sample set c_2010 = t.res
                    from (select ssid_in, sum(case when out_year - in_year >= 0 and out_year <= 2010 and in_year <= 2010 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.ssid_in) as t where t.ssid_in = sspublications_sample.ssid;
'''
writer.execute_custom_query(query=query_c_2010)

CPU times: user 1.91 ms, sys: 15 µs, total: 1.93 ms
Wall time: 19.4 s


In [20]:
%%time 
query_c_2020 = '''
                alter table sspublications_sample add column if not exists c_2020 int default 0;
                update sspublications_sample set c_2020 = t.res
                    from (select ssid_in, sum(case when out_year - in_year >= 0 and out_year <= 2020 and in_year <= 2020 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.ssid_in) as t where t.ssid_in = sspublications_sample.ssid;
'''
writer.execute_custom_query(query=query_c_2020)

CPU times: user 1.59 ms, sys: 199 µs, total: 1.79 ms
Wall time: 20.9 s


In [5]:
def hindex(citations):
    ls = sorted(citations, reverse=True)
    for h in range(len(ls)):
        if h + 1 > ls[h]:
            return h
    return len(ls)

In [6]:
%%time
query_id = '''select author_id from authors;'''
query_res = loader.custom_query(query=query_id) 
data_hindex = {item[0]: [0, 0, 0, 0, 0, 0] for item in query_res}

for idx, year in enumerate(range(1970, 2021, 10)):
    q = f'''select author_id, c_{year} from authors_papers left join sspublications_sample on authors_papers.ssid = sspublications_sample.ssid'''
    tmp_data = loader.custom_query(query=q)
    citations = defaultdict(list)
    for a_id, cit in tmp_data:
        citations[a_id].append(cit)

    result = {a_id: hindex(citations[a_id]) for a_id in citations}
    for a_id in result:
        data_hindex[a_id][idx] = result[a_id]

CPU times: user 33.5 s, sys: 2.15 s, total: 35.6 s
Wall time: 58.1 s


In [15]:
def insert_hindex(data):
    try:
        ls = []
        for a_id in sorted(list(data.keys())):
            ls.append(f'({a_id}, {", ".join(map(str, data[a_id]))})')
        query = f'''alter table authors add column if not exists h_1970 int default 0;
                    alter table authors add column if not exists h_1980 int default 0;
                    alter table authors add column if not exists h_1990 int default 0;
                    alter table authors add column if not exists h_2000 int default 0;
                    alter table authors add column if not exists h_2010 int default 0;
                    alter table authors add column if not exists h_2020 int default 0;
                    insert into temp_table values {', '.join(ls)};
                    update authors set h_1970 = t.h_1970, h_1980 = t.h_1980, h_1990 = t.h_1990, h_2000 = t.h_2000, h_2010 = t.h_2010, h_2020 = t.h_2020 
                        from (select * from temp_table) as t where t.a_id = authors.author_id;
                    drop table temp_table;
                    create table if not exists temp_table (a_id bigint, h_1970 int, h_1980 int, h_1990 int, h_2000 int, h_2010 int, h_2020 int);
                    '''
        writer.check_connection()
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)

In [ ]:
%%time
insert_hindex(data_hindex)

### Table for venues and journals

In [6]:
%%time
q = '''select authors_papers.author_id, authors_papers.ssid, ven, jour from authors_papers left join (SELECT ssid, aux::json->'venue' as ven, aux::json#>'{journal, name}' as jour
                                        FROM sspublications_sample) as t on authors_papers.ssid = t.ssid'''
journals_and_venues = loader.custom_query(q)

CPU times: user 14.6 s, sys: 928 ms, total: 15.5 s
Wall time: 4min 6s


In [15]:
from difflib import SequenceMatcher



def process_venues_authors_papers(ls):
    items = defaultdict(set)
    
    for a_id, p_id, ven, jour in ls:
        ven, jour = ' '.join(ven.strip().split()).lower(), ' '.join(jour.strip().split()).lower()
        if SequenceMatcher(None, ven, jour).ratio() > 0.5:
            key = min(ven, jour, key=len)
            items[key].add((a_id, p_id))
        else:
            items[ven].add((a_id, p_id))
            items[jour].add((a_id, p_id))
    
    del items['']
    
    data_for_vpa_table = []
    data_for_venue_table = []
    for idx, key in enumerate(items, start=1):
        for a_id, p_id in items[key]:
            data_for_vpa_table.append((idx, a_id, p_id))
        data_for_venue_table.append((idx, key))
    return data_for_vpa_table, data_for_venue_table

In [16]:
%%time
vpa, venues = process_venues_authors_papers(journals_and_venues)
len(vpa), len(venues)

CPU times: user 1min 23s, sys: 66.7 ms, total: 1min 23s
Wall time: 1min 23s


(2628957, 70594)

In [17]:
def create_and_insert_vpa(data):
    try:
        ls = [f'''({v_id}, {a_id}, '{p_id}')''' for v_id, a_id, p_id in data]
        query = f'''create table if not exists venues_papers_authors (venue_id int, author_id bigint, ssid varchar(40));
                    insert into venues_papers_authors values {', '.join(ls)};
                    create index if not exists vpa_venue_id on venues_papers_authors (venue_id);
                    create index if not exists vpa_author_id on venues_papers_authors (author_id);
                    create index if not exists vpa_paper_id on venues_papers_authors (ssid);'''
        writer.check_connection()
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)


def create_and_insert_venues(data):
    try:
        ls = [f'''({v_id}, '{v_name.replace('"', '').replace("'", '')}')''' for v_id, v_name in data]
        query = f'''create table if not exists venues (venue_id int, venue_name varchar(256));
                    insert into venues values {', '.join(ls)};
                    create index if not exists venues_venue_id on venues (venue_id);'''
        writer.check_connection()
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)

In [18]:
%%time
create_and_insert_vpa(data=vpa)

CPU times: user 1.24 s, sys: 265 ms, total: 1.5 s
Wall time: 21.7 s


In [19]:
%%time
create_and_insert_venues(data=venues)

CPU times: user 45.9 ms, sys: 26 µs, total: 45.9 ms
Wall time: 344 ms


### Num of papers for venue

In [20]:
%%time
query_total_papers_v = '''alter table venues add column if not exists total_papers int default 0;
            update venues set total_papers = f.res from (select t.venue_id, count(*) as res 
                from (select distinct vpa.ssid, venue_id, c_all 
                    from venues_papers_authors as vpa left join sspublications_sample as ss on vpa.ssid=ss.ssid) as t 
                        group by t.venue_id) as f where f.venue_id = venues.venue_id;'''
writer.execute_custom_query(query=query_total_papers_v)

CPU times: user 1.51 ms, sys: 0 ns, total: 1.51 ms
Wall time: 4.21 s


### Total citations

In [21]:
%%time
query_total_cit_v = '''alter table venues add column if not exists total_citations int default 0;
            update venues set total_citations = f.res from (select t.venue_id, sum(t.c_all) as res 
                from (select distinct vpa.ssid, venue_id, c_all 
                    from venues_papers_authors as vpa left join sspublications_sample as ss on vpa.ssid=ss.ssid) as t 
                        group by t.venue_id) as f where f.venue_id = venues.venue_id;'''
writer.execute_custom_query(query=query_total_cit_v)

CPU times: user 1.17 ms, sys: 333 µs, total: 1.5 ms
Wall time: 4.03 s


### Avg citations per paper


In [22]:
%%time
query_avg_cit_v = '''alter table venues add column if not exists avg_citations_paper decimal default 0.0;
            update venues set avg_citations_paper = f.res from (select t.venue_id, avg(t.c_all) as res 
                from (select distinct vpa.ssid, venue_id, c_all 
                    from venues_papers_authors as vpa left join sspublications_sample as ss on vpa.ssid=ss.ssid) as t 
                        group by t.venue_id) as f where f.venue_id = venues.venue_id;'''
writer.execute_custom_query(query=query_avg_cit_v)

CPU times: user 0 ns, sys: 1.48 ms, total: 1.48 ms
Wall time: 4.07 s


### Avg citations per year

In [23]:
%%time
query_avg_cit_v_y = '''alter table venues add column if not exists avg_citations_year decimal default 0.0;
            update venues set avg_citations_year = f.res from (select venue_id, avg(tmp) as res 
            from (select venue_id, sum(c_all) as tmp 
                from (select distinct vpa.ssid, year, venue_id, c_all 
                    from venues_papers_authors as vpa left join sspublications_sample as ss on ss.ssid = vpa.ssid) as sub_table
                                        group by (venue_id, year)) as top_table 
                                        group by venue_id) as f where f.venue_id = venues.venue_id;'''
writer.execute_custom_query(query=query_avg_cit_v_y)

CPU times: user 1.45 ms, sys: 23 µs, total: 1.48 ms
Wall time: 4.46 s


### Avg papers per year

In [24]:
%%time
query_avg_papers_y = '''alter table venues add column if not exists avg_papers_year decimal default 0.0;
            update venues set avg_papers_year = f.res from (select venue_id, avg(tmp) as res 
            from (select venue_id, count(*) as tmp 
                from (select distinct vpa.ssid, year, venue_id, c_all 
                    from venues_papers_authors as vpa left join sspublications_sample as ss on ss.ssid = vpa.ssid) as sub_table
                                        group by (venue_id, year)) as top_table 
                                        group by venue_id) as f where f.venue_id = venues.venue_id;'''
writer.execute_custom_query(query=query_avg_papers_y)

CPU times: user 1.39 ms, sys: 22 µs, total: 1.41 ms
Wall time: 4.46 s


In [ ]:
# num of papers
# select t.venue_id, count(*) as res from (select distinct vpa.ssid, venue_id, c_all from venues_papers_authors as vpa left join sspublications_sample as ss on vpa.ssid=ss.ssid) as t group by t.venue_id order by res desc;
# avg citations
#  select t.venue_id, avg(t.c_all) from (select distinct vpa.ssid, venue_id, c_all from venues_papers_authors as vpa left join sspublications_sample as ss on vpa.ssid=ss.ssid) as t group by t.venue_id;

# total citations
#  select t.venue_id, sum(t.c_all) from (select distinct vpa.ssid, venue_id, c_all from venues_papers_authors as vpa left join sspublications_sample as ss on vpa.ssid=ss.ssid) as t group by t.venue_id;

## Papers features

### Num of authors

In [11]:
!python3 -m pip install langdetect

     |████████████████████████████████| 981 kB 1.6 MB/s eta 0:00:01
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=14f34cb6e170c837bf26eefd78150c570613ee54c5ab152966dab430c0d9834d
  Stored in directory: /home/student/.cache/pip/wheels/c5/96/8a/f90c59ed25d75e50a8c10a1b1c2d4c402e4dacfa87f3aff36a
Successfully built langdetect


In [5]:
%%time
query_num_of_authors = '''alter table sspublications_sample add column if not exists num_of_authors int default 0;
                            update sspublications_sample set num_of_authors = t.res from (select ssid, count(*) as res
                            from authors_papers group by ssid) as t where t.ssid = sspublications_sample.ssid;'''
writer.execute_custom_query(query=query_num_of_authors)

CPU times: user 3.45 ms, sys: 0 ns, total: 3.45 ms
Wall time: 24.5 s


### Num of words in title and abstract (and languages)
(keyword everywhere is none)

In [6]:
%%time
query_titles = '''select ssid, title from sspublications_sample;'''
query_abstracts = '''select ssid, abstract from sspublications_sample;'''

titles = loader.custom_query(query=query_titles)
abstracts = loader.custom_query(query=query_abstracts)

CPU times: user 2.13 s, sys: 1.43 s, total: 3.56 s
Wall time: 10.4 s


In [7]:
from langdetect import detect


def process_text(data):
    ls = []
    for idx, (key, text) in enumerate(data):
        if text:
            tmp = ''.join([item for item in text.strip() if item.isalpha() or item == ' ']).split()
            ls.append((key, len(tmp)))
        else:
            ls.append((key, 0))
    return ls


def lang_detect(data):
    ls = []
    for key, item in data:
        if item:
            try:
                res = detect(item)
                ls.append((key, res))
            except Exception as e:
                ls.append((key, None))
    return ls



In [19]:
%%time

data_titles = process_text(titles)
data_abstracts = process_text(abstracts)

title_lang = lang_detect(titles)
abs_lang = lang_detect(abstracts)

CPU times: user 2h 52min 43s, sys: 5min 40s, total: 2h 58min 23s
Wall time: 2h 58min 23s


In [8]:
%%time
data_titles = process_text(titles)
data_abstracts = process_text(abstracts)

CPU times: user 1min 17s, sys: 172 ms, total: 1min 17s
Wall time: 1min 17s


In [25]:
# with open('tmp_data/abs_langs.txt', 'w') as f:
#     ls = [f'{key} {val}\n' for key, val in abs_lang]
#     f.writelines(ls)

# with open('tmp_data/title_langs.txt', 'w') as f:
#     ls = [f'{key} {val}\n' for key, val in title_lang]
#     f.writelines(ls)

In [15]:
def insert_num_of_words(data_titles, data_abs):
    try:
        ls = []
        for (key_t, val_t), (key_a, val_a) in zip(data_titles, data_abs):
            assert key_t == key_a, 'Keys must be the same'
            ls.append(f'''('{key_t}', {val_t}, {val_a})''')
        
        query = f'''alter table sspublications_sample add column if not exists num_words_title int default 0;
                    alter table sspublications_sample add column if not exists num_words_abs int default 0;
                    create table if not exists temp_table (ssid varchar(40), num_words_title int, num_words_abs int);
                    insert into temp_table values {','.join(ls)};
                    update sspublications_sample set num_words_title = t.num_words_title, num_words_abs = t.num_words_abs  
                        from (select ssid, num_words_title, num_words_abs from temp_table) as t where t.ssid = sspublications_sample.ssid;
                    drop table temp_table;
                    '''
        writer.check_connection()
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)
        

In [16]:
%%time
insert_num_of_words(data_titles, data_abstracts)

CPU times: user 511 ms, sys: 60.2 ms, total: 571 ms
Wall time: 7.69 s


In [9]:
with open('tmp_data/title_langs.txt') as f:
    ls = [item.strip().split() for item in f.readlines()]

In [10]:
# есть пропуски в id для заголовков; поэтому добавим недостающие id и дефолтные значения для них
keys = set([key for key, _ in data_titles]) - set([key for key, _ in ls])
for key in keys:
    ls.append([key, None])

In [16]:
def insert_lang(data):
    try:
        ls = []
        for key, val in data:
            ls.append(f'''('{key}', '{val if val is not None else "none"}')''')
        
        query = f'''alter table sspublications_sample add column if not exists lang varchar(10) default NULL;
                    create table if not exists temp_table (ssid varchar(40), lang varchar(10));
                    insert into temp_table values {','.join(ls)};
                    update sspublications_sample set lang = t.lang
                        from (select ssid, lang from temp_table) as t where t.ssid = sspublications_sample.ssid;
                    drop table temp_table;
                    '''
        writer.check_connection()
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)

In [17]:
%%time
insert_lang(ls)

CPU times: user 294 ms, sys: 60.2 ms, total: 354 ms
Wall time: 9.21 s
